In [1]:
# extract
import numpy as np
from numpy import linalg as LA

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input

class VGGNet:
    def __init__(self):
        self.input_shape = (224, 224, 3)
        self.weight = 'imagenet'
        self.pooling = 'max'
        self.model = VGG16(weights = self.weight, input_shape = (self.input_shape[0], self.input_shape[1], self.input_shape[2]), pooling = self.pooling, include_top = False)
        self.model.predict(np.zeros((1, 224, 224 , 3)))

    def extract_feat(self, img_path):
        img = image.load_img(img_path, target_size=(self.input_shape[0], self.input_shape[1]))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        feat = self.model.predict(img)
        norm_feat = feat[0]/LA.norm(feat[0])
        return norm_feat


In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
import os
import h5py
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [4]:
images_path ="/content/gdrive/MyDrive/ml/train+vali/"

model = VGGNet()
path = "/content/gdrive/MyDrive/ml/train+vali/"

feats = []
names = []

# for im in os.listdir(path):
#     X = model.extract_feat(path+im)
#     feats.append(X)
#     names.append(im)
for root, dirs, files in os.walk(images_path):
    for f in files:
      full_path = os.path.join(root, f)
      X = model.extract_feat(full_path)
      feats.append(X)
      rel_path = os.path.relpath(full_path, images_path)
      names.append(rel_path)
feats = np.array(feats)
output = "VGG16Features.h5"

h5f = h5py.File(output, 'w')
h5f.create_dataset('dataset_1', data = feats)
h5f.create_dataset('dataset_2', data = np.bytes_(names))
h5f.close()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 866ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 660ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 926ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 612ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 566ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 537ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 544ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 566ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 535ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 564ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 549ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 556ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 567ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 536ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 564ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 565ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 537ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 560ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 694ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 994ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 898ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 587ms/step
1/1 ━━━━━━━━━━━━━━━

In [5]:
# เปิดไฟล์ HDF5
with h5py.File('VGG16Features.h5', 'r') as h5f:
    # ตรวจสอบ keys ในไฟล์
    print("Keys in the HDF5 file:", list(h5f.keys()))

    # ตรวจสอบข้อมูลใน dataset_2 ก่อน
    dataset_2 = h5f['dataset_2']
    print(f"Dataset_2 info: {dataset_2}")

    # ถ้า dataset_2 เป็น scalar หรือเป็น array ที่ไม่สามารถ slice ได้
    if dataset_2.shape == ():
        # ถ้าเป็น scalar, จะดึงค่ามาเป็นค่าคงที่
        print("Dataset_2 is scalar, can't slice it.")
        names = dataset_2[()]
    else:
        # ถ้าเป็น array, สามารถ slice ได้
        names = dataset_2[:]

    print(f"Names of images (dataset_2): {names}")

    # ตัวอย่างข้อมูลแรกในแต่ละ dataset
    feats = h5f['dataset_1'][:]
    print(f"First feature vector: {feats[0]}")


Keys in the HDF5 file: ['dataset_1', 'dataset_2']
Dataset_2 info: <HDF5 dataset "dataset_2": shape (2954,), type "|S191">
Names of images (dataset_2): [b'Machu Pichu/10.jpg' b'Machu Pichu/17.jpg' b'Machu Pichu/13.jpg' ...
 b'Venezuela Angel Falls/383.jpg' b'Venezuela Angel Falls/389.jpg'
 b'Venezuela Angel Falls/403.jpg']
First feature vector: [4.86480370e-02 6.81978911e-02 6.13538101e-02 0.00000000e+00
 1.31441774e-02 7.61607895e-03 9.23851505e-03 2.59215888e-02
 0.00000000e+00 7.44269192e-02 9.31273215e-03 0.00000000e+00
 1.52057726e-02 0.00000000e+00 5.06892335e-03 1.63828058e-03
 8.05180706e-03 0.00000000e+00 8.73604976e-03 1.75791364e-02
 1.08255453e-01 1.53482100e-02 3.41890310e-03 1.42055759e-02
 0.00000000e+00 1.04184166e-01 0.00000000e+00 2.25204322e-02
 4.42142822e-02 0.00000000e+00 3.26213352e-02 2.19486579e-02
 2.21357923e-02 0.00000000e+00 3.64870648e-04 4.18974012e-02
 0.00000000e+00 2.31836177e-02 8.84243771e-02 6.18689843e-02
 2.34348569e-02 3.82612390e-03 2.23773345e-0

In [6]:
with h5py.File('VGG16Features.h5', 'r') as h5f:
    # ตรวจสอบ keys ในไฟล์
    print("Keys in the HDF5 file:", list(h5f.keys()))

    # โหลด dataset_1 (ฟีเจอร์)
    feats = h5f['dataset_1'][:]
    print(f"Shape of features dataset (dataset_1): {feats.shape}")

Keys in the HDF5 file: ['dataset_1', 'dataset_2']
Shape of features dataset (dataset_1): (2954, 512)


In [7]:
 with h5py.File('VGG16Features.h5', 'r') as h5f:
      # โหลด dataset_2 (ชื่อภาพ)
    names = h5f['dataset_2'][:]
    print(f"Names of images (dataset_2): {names}")

Names of images (dataset_2): [b'Machu Pichu/10.jpg' b'Machu Pichu/17.jpg' b'Machu Pichu/13.jpg' ...
 b'Venezuela Angel Falls/383.jpg' b'Venezuela Angel Falls/389.jpg'
 b'Venezuela Angel Falls/403.jpg']


In [8]:
 with h5py.File('VGG16Features.h5', 'r') as h5f:
    # ตัวอย่างข้อมูลแรกในแต่ละ dataset
    print(f"First feature vector: {feats[0]}")
    print(f"First image name: {names[0]}")

First feature vector: [4.86480370e-02 6.81978911e-02 6.13538101e-02 0.00000000e+00
 1.31441774e-02 7.61607895e-03 9.23851505e-03 2.59215888e-02
 0.00000000e+00 7.44269192e-02 9.31273215e-03 0.00000000e+00
 1.52057726e-02 0.00000000e+00 5.06892335e-03 1.63828058e-03
 8.05180706e-03 0.00000000e+00 8.73604976e-03 1.75791364e-02
 1.08255453e-01 1.53482100e-02 3.41890310e-03 1.42055759e-02
 0.00000000e+00 1.04184166e-01 0.00000000e+00 2.25204322e-02
 4.42142822e-02 0.00000000e+00 3.26213352e-02 2.19486579e-02
 2.21357923e-02 0.00000000e+00 3.64870648e-04 4.18974012e-02
 0.00000000e+00 2.31836177e-02 8.84243771e-02 6.18689843e-02
 2.34348569e-02 3.82612390e-03 2.23773345e-02 3.23767997e-02
 2.90405229e-02 9.19637308e-02 9.27738659e-03 2.51407325e-02
 6.56630518e-03 1.06148019e-01 0.00000000e+00 0.00000000e+00
 3.15035656e-02 5.88305593e-02 1.27897051e-03 3.85720246e-02
 2.45824223e-03 3.27581391e-02 3.47331129e-02 0.00000000e+00
 3.32647525e-02 2.46291012e-02 2.79734866e-03 5.97158913e-03
 3